# SofaScore — extrair coordenadas da partida

Cole a **URL da partida** na célula de configuração e execute todas as células.

O notebook gera um **CSV único** com passes e conduções de **todos os jogadores**, incluindo:
- coordenadas (`x`, `y`, `end_x`, `end_y`)
- jogador, time, posição
- tempo do evento e demais campos retornados pela API

> Se aparecer erro **403**, altere `MODE = "browser"` (precisa do Google Chrome instalado).

In [ ]:
import sys
from pathlib import Path

# Garante que o Python encontre extract_sofascore_match_events.py
# mesmo se o Jupyter iniciar em outra pasta.


def find_project_root(marker: str = "extract_sofascore_match_events.py") -> Path:
    candidates = [Path.cwd(), *Path.cwd().parents]
    for base in candidates:
        if (base / marker).exists():
            return base.resolve()
    raise FileNotFoundError(
        "Não encontrei extract_sofascore_match_events.py.\n"
        "Abra o notebook na pasta do repositório (onde está o .py) "
        "ou faça git pull da branch cursor/sofascore-extract-events-88ee."
    )


PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"Pasta do projeto: {PROJECT_ROOT}")

# Execute uma vez se faltar alguma dependência:
# %pip install -q curl_cffi pandas undetected-chromedriver selenium beautifulsoup4 ipykernel

In [ ]:
import pandas as pd

from extract_sofascore_match_events import extract_match_events, parse_match_id

# ========== CONFIGURAÇÃO — edite aqui ==========
MATCH_URL = "https://www.sofascore.com/football/match/argentina-austria/tUbsuWb#id:15186502"

# "auto" tenta curl_cffi e cai para Chrome se receber 403
# use "browser" se "auto" falhar
MODE = "auto"

# nome do CSV final (passes + conduções de todos os jogadores)
OUTPUT_CSV = PROJECT_ROOT / "coordenadas_partida.csv"
# ===============================================

match_id = parse_match_id(MATCH_URL)
output_dir = PROJECT_ROOT / "notebook_output" / str(match_id)
output_dir

In [ ]:
result = extract_match_events(
    MATCH_URL,
    output_dir,
    mode=MODE,
    delay=0.8,
    save_raw=False,
)

df = result["all_events"].copy()
print(f"Jogadores no lineup: {len(result['lineups'])}")
print(f"Passes: {len(result['passes'])} | Conduções: {len(result['ball_carries'])}")
print(f"Total de eventos: {len(df)}")
df.head()

In [ ]:
# Colunas principais de coordenadas (o CSV salva todas as colunas disponíveis)
coord_cols = [
    "match_id", "category", "player_name", "team", "side", "player_position",
    "x", "y", "end_x", "end_y", "time", "timeSeconds", "outcome", "length", "progressive",
]
preview_cols = [c for c in coord_cols if c in df.columns]
df[preview_cols].head(10)

In [ ]:
csv_path = Path(OUTPUT_CSV)
df.to_csv(csv_path, index=False)

print(f"CSV salvo em: {csv_path}")
print(f"Linhas: {len(df)} | Colunas: {len(df.columns)}")
df["category"].value_counts()